# NBA Rules RAG — Colab Demo

**Multimodal pipeline**: YouTube clip → frame extraction → VLM narration → RAG over NBA rulebook → grounded LLM answer with rule citations.

### Pipeline
```
YouTube URL + timestamps + question
        ↓
  Frame extraction (yt-dlp + OpenCV)
        ↓
  VLM narration  (any OpenAI-compatible vision model)
        ↓
  Query building from VLM output + question
        ↓
  FAISS retrieval over NBA rulebook embeddings
        ↓
  LLM reasoning  (any OpenAI-compatible text model)
        ↓
  Answer with rule citations
```

**Requirements**: An API key for your chosen inference provider and the NBA rulebook PDF.

> Run cells top-to-bottom. Colab GPU runtime is optional — embedding and inference run on CPU for small models.

## 0 — Install dependencies

In [ ]:
# Install system dependency for OpenCV
!apt-get install -qq libgl1

# Clone repo and install Python package
import os, pathlib

REPO_DIR = pathlib.Path("/content/nba-rules-rag")

if not REPO_DIR.exists():
    !git clone https://github.com/YOUR_USERNAME/nba-rules-rag.git {REPO_DIR}

%cd {REPO_DIR}

!pip install -q -r requirements.txt
!pip install -q -e .

print("✓ Dependencies installed")

## 1 — Authenticate and configure

Set your API key and model names. The pipeline accepts `OPENAI_API_KEY` (for OpenAI or any compatible provider) or `HF_TOKEN` (for HuggingFace).  
Also set `VLM_MODEL` and `REASONER_MODEL` to the model IDs you want to use.  
In Colab, store secrets in the Secrets panel (🔑 icon in the left sidebar) and the cell reads them automatically.

In [ ]:
import os

# ── API key ─────────────────────────────────────────────────────────────────
# The pipeline checks OPENAI_API_KEY first, then HF_TOKEN.
# Set whichever matches your inference provider.
try:
    from google.colab import userdata
    # Load from Colab Secrets — add keys via the 🔑 panel on the left.
    for key in ("OPENAI_API_KEY", "HF_TOKEN"):
        try:
            os.environ[key] = userdata.get(key)
        except Exception:
            pass
except ImportError:
    pass  # not in Colab

# Fall back to manual values if the secrets are not set.
# Replace these placeholders with your real keys.
if not os.environ.get("OPENAI_API_KEY") and not os.environ.get("HF_TOKEN"):
    os.environ["OPENAI_API_KEY"] = "sk-YOUR_KEY_HERE"
    print("⚠ API key set manually — replace with your real key")

# ── Model names ──────────────────────────────────────────────────────────────
# Set these to the model IDs you want to use.
os.environ.setdefault("VLM_MODEL", "gpt-4o")           # vision model
os.environ.setdefault("REASONER_MODEL", "gpt-4o")      # text reasoning model

# ── API base (optional) ───────────────────────────────────────────────────────
# Defaults to https://api.openai.com/v1/chat/completions
# For HuggingFace: os.environ["VLM_API_BASE"] = "https://router.huggingface.co/v1/chat/completions"
# For Azure:       os.environ["VLM_API_BASE"] = "https://YOUR_RESOURCE.openai.azure.com/..."

print("VLM_MODEL      :", os.environ.get("VLM_MODEL"))
print("REASONER_MODEL :", os.environ.get("REASONER_MODEL"))
print("API key set    :", bool(os.environ.get("OPENAI_API_KEY") or os.environ.get("HF_TOKEN")))

## 2 — Upload the NBA rulebook PDF and build the vector index

Download the official NBA rulebook PDF and upload it to Colab, **or** point `PDF_PATH` at a path in your Google Drive.  
This step only needs to run once — the FAISS index is saved to `data/vector_store/`.

In [ ]:
import pathlib

# ── Option A: upload from local machine ──────────────────────────────────────
# Uncomment to trigger the Colab file-upload widget:
# from google.colab import files
# uploaded = files.upload()  # select your PDF
# PDF_PATH = pathlib.Path(next(iter(uploaded)))

# ── Option B: path inside Google Drive ───────────────────────────────────────
# from google.colab import drive
# drive.mount('/content/drive')
# PDF_PATH = pathlib.Path('/content/drive/MyDrive/nba-rulebook.pdf')

# ── Option C: download directly (public URL if available) ────────────────────
# !wget -q -O docs/rulebook.pdf "https://your-public-url/rulebook.pdf"
# PDF_PATH = pathlib.Path('docs/rulebook.pdf')

# Update this to your chosen path:
PDF_PATH = pathlib.Path("docs/2023-24-NBA-Season-Official-Playing-Rules.pdf")

print(f"PDF path set to: {PDF_PATH}")
print(f"PDF exists: {PDF_PATH.exists()}")

In [ ]:
import pathlib
from scripts.build_rulebook_index import build_rulebook_index

VECTOR_STORE_DIR = pathlib.Path("data/vector_store")

# Skip rebuild if index already exists.
if (VECTOR_STORE_DIR / "faiss.index").exists():
    print("✓ Vector index already exists — skipping build.")
    print(f"  Store: {VECTOR_STORE_DIR}")
else:
    print("Building rulebook index (this may take ~1–2 min on CPU)…")
    meta = build_rulebook_index(
        pdf_path=PDF_PATH,
        output_dir=VECTOR_STORE_DIR,
        enable_chunking=True,
        chunk_size_words=180,
        overlap_words=40,
    )
    print(f"✓ Index built: {meta['n_chunks']} chunks, saved to {VECTOR_STORE_DIR}")

## 3 — Set your inputs

Provide a YouTube URL, a time window (HH:MM:SS, MM:SS, or SS), and your question.

In [ ]:
# ── Inputs ────────────────────────────────────────────────────────────────────
YOUTUBE_URL = "https://www.youtube.com/watch?v=9fFWawcJXUw"
START_TIME  = "0:21"
END_TIME    = "0:29"
QUESTION    = "Was the player in the blue uniform traveling?"

# Number of top rulebook chunks to retrieve
RETRIEVAL_TOP_K = 5

print(f"URL   : {YOUTUBE_URL}")
print(f"Clip  : {START_TIME} → {END_TIME}")
print(f"Query : {QUESTION}")

## 4 — Extract frames from the YouTube clip

In [ ]:
import matplotlib.pyplot as plt
import math
from nba_rules_rag.youtube_utils import parse_youtube_url, validate_interval
from nba_rules_rag.frame_extraction import extract_frames_from_youtube

video_id = parse_youtube_url(YOUTUBE_URL)
start_sec, end_sec = validate_interval(START_TIME, END_TIME)

print(f"Video ID : {video_id}")
print(f"Interval : {start_sec:.2f}s → {end_sec:.2f}s  ({end_sec - start_sec:.1f}s clip)")
print("Downloading and extracting frames…")

frames = extract_frames_from_youtube(video_id, start_sec, end_sec)
n = len(frames)
print(f"✓ Extracted {n} frame(s)")

# Display frame grid
cols = min(n, 4)
rows = math.ceil(n / cols)
fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 3 * rows))
axes = [axes] if n == 1 else list(plt.gcf().get_axes())
step = (end_sec - start_sec) / max(n - 1, 1)
for i, (ax, frame) in enumerate(zip(axes, frames)):
    ax.imshow(frame)
    ax.set_title(f"F{i+1:02d}  t={start_sec + i*step:.2f}s", fontsize=9)
    ax.axis("off")
for ax in axes[n:]:
    ax.axis("off")
plt.suptitle(f"Extracted frames — {YOUTUBE_URL}", fontsize=11)
plt.tight_layout()
plt.show()

## 5 — VLM narration

Send the ordered frames + question to your configured VLM (`VLM_MODEL` env var).  
The model returns a structured JSON describing the play, per-frame observations, travel signals, and evidence.

In [ ]:
import json
from nba_rules_rag.vlm_describer import describe_frames_with_vlm

# Estimate per-frame timestamps
step = (end_sec - start_sec) / max(len(frames) - 1, 1)
frame_timestamps_sec = [start_sec + i * step for i in range(len(frames))]

print("Calling VLM…")
vlm_result = describe_frames_with_vlm(
    frames,
    QUESTION,
    frame_timestamps_sec=frame_timestamps_sec,
)

print("\n── Play Narration ────────────────────────────────────────────────")
print(vlm_result.get("play_narration", "(none)"))

print("\n── Sequence Summary ──────────────────────────────────────────────")
print(vlm_result.get("sequence_summary", "(none)"))

print("\n── Travel Features ───────────────────────────────────────────────")
tf = vlm_result.get("travel_features", {})
for k, v in (tf or {}).items():
    print(f"  {k}: {v}")

print("\n── Raw VLM JSON ──────────────────────────────────────────────────")
print(json.dumps(vlm_result, indent=2))

## 6 — Embed and retrieve relevant NBA rulebook chunks

In [ ]:
from nba_rules_rag.query_builder import build_retrieval_query
from nba_rules_rag.retriever import retrieve_rulebook_chunks

retrieval_query = build_retrieval_query(vlm_result, fallback_question=QUESTION)
print("Retrieval query:")
print(retrieval_query)

print(f"\nSearching top-{RETRIEVAL_TOP_K} chunks…")
retrieved_chunks = retrieve_rulebook_chunks(
    retrieval_query,
    top_k=RETRIEVAL_TOP_K,
    vector_store_dir=VECTOR_STORE_DIR,
)

print(f"\n✓ Retrieved {len(retrieved_chunks)} chunk(s)\n")
for i, chunk in enumerate(retrieved_chunks, 1):
    rule_id   = chunk.get('rule_id', '')
    title     = chunk.get('section_title', '')
    score     = chunk.get('score', 0)
    text_snip = chunk.get('text', '')[:300]
    print(f"[{i}] Rule {rule_id} — {title}  (score: {score:.4f})")
    print(f"    {text_snip}…\n")

## 7 — LLM reasoning with rule citations

Pass the VLM narration + retrieved rulebook chunks to your configured reasoning model (`REASONER_MODEL` env var).  
The model must state which specific NBA rules apply and give a grounded ruling.

In [ ]:
import json
from nba_rules_rag.reasoner import reason_with_rules

print("Calling LLM reasoner…")
reasoner_result = reason_with_rules(
    QUESTION,
    vlm_result,
    retrieved_chunks,
)

print("\n═══════════════════════════════════════════════════════════════")
print("  RULING:", reasoner_result.get("ruling", "unknown").upper())
print("═══════════════════════════════════════════════════════════════")
print("\nAnswer:")
print(reasoner_result.get("answer", ""))

print("\nReasoning:")
print(reasoner_result.get("reasoning", ""))

print("\nRelevant rules cited:")
for rule in reasoner_result.get("relevant_rules", []):
    print(f"  • {rule.get('rule_id','')} {rule.get('section','')}: {rule.get('citation','')}")
    print(f"    Relevance: {rule.get('relevance','')}")

print(f"\nConfidence: {reasoner_result.get('confidence', 'unknown')}")
uncertainties = reasoner_result.get("uncertainties", [])
if uncertainties:
    print("Uncertainties:")
    for u in uncertainties:
        print(f"  - {u}")

print("\n── Full JSON ─────────────────────────────────────────────────────")
print(json.dumps(reasoner_result, indent=2))

## 8 — Full end-to-end pipeline (convenience wrapper)

The `run_pipeline` function in [src/nba_rules_rag/pipeline.py](../src/nba_rules_rag/pipeline.py) runs all steps above in one call.

In [ ]:
import json
from nba_rules_rag.pipeline import run_pipeline

result = run_pipeline(
    youtube_url=YOUTUBE_URL,
    start_time=START_TIME,
    end_time=END_TIME,
    question=QUESTION,
    retrieval_top_k=RETRIEVAL_TOP_K,
)

print("Status :", result["status"])
print("Frames :", result["n_frames"])
print("VLM err:", result["vlm_error"])
print("RAG err:", result["retrieval_error"])
print("LLM err:", result["reasoner_error"])
print("\nRuling :", (result.get("reasoner_result") or {}).get("ruling", "n/a"))
print("Answer :", (result.get("reasoner_result") or {}).get("answer", "n/a"))